In [0]:
# Load airports.csv into DataFrame: df_airports
df_airports = (
    spark.read
         .option("header", "true")
         .option("delimiter", ",")
         .csv("/Volumes/vita_prac/vita_sche/vita_vol/airports.csv")
)

# Load runways.csv into DataFrame: df_runways
df_runways = (
    spark.read
         .option("header", "true")
         .option("delimiter", ",")
         .csv("/Volumes/vita_prac/vita_sche/vita_vol/runways.csv")
)

# Verify data
df_airports.show(5)
df_runways.show(5)

+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------+--------+
|    id|ident|         type|                name|     latitude_deg|     longitude_deg|elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|           home_link|wikipedia_link|keywords|
+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------+--------+
|  6523|  00A|     heliport|   Total RF Heliport|        40.070985|        -74.933689|          11|       NA|         US|     US-PA|    Bensalem|               no|     NULL|     NULL|    K00A|       00A|https://www.pennd...|          NULL|    NULL|
|323

In [0]:
from pyspark.sql.functions import col

# Cast numerical columns in df_airports
df_airports = (
    df_airports
    .withColumn("latitude_deg", col("latitude_deg").cast("double"))
    .withColumn("longitude_deg", col("longitude_deg").cast("double"))
    .withColumn("elevation_ft", col("elevation_ft").cast("int"))
)

# Cast numerical columns in df_runways
df_runways = (
    df_runways
    .withColumn("length_ft", col("length_ft").cast("int"))
    .withColumn("width_ft", col("width_ft").cast("int"))
)

# Verify schema
df_airports.printSchema()
df_runways.printSchema()

root
 |-- id: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- airport_ref: string (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = tru

In [0]:
# Print schema of both DataFrames
df_airports.printSchema()
df_runways.printSchema()

# Print row counts
print("df_airports row count:", df_airports.count())
print("df_runways row count:", df_runways.count())

root
 |-- id: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- airport_ref: string (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = tru

In [0]:
# Inner join df_airports and df_runways
df_combined = (
    df_airports.join(
        df_runways,
        df_airports["ident"] == df_runways["airport_ident"],
        "inner"
    )
)

# Preview joined data
df_combined.show(5)

+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------+--------+------+-----------+-------------+---------+--------+-------+-------+------+--------+---------------+----------------+---------------+---------------+-------------------------+--------+---------------+----------------+---------------+---------------+-------------------------+
|    id|ident|         type|                name|     latitude_deg|     longitude_deg|elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|           home_link|wikipedia_link|keywords|    id|airport_ref|airport_ident|length_ft|width_ft|surface|lighted|closed|le_ident|le_latitude_deg|le_longitude_deg|le_elevation_ft|le_heading_degT|le_displaced_threshold_ft|he_ident|he_latitude_deg|he_longitude_deg|he_elevatio

In [0]:
from pyspark.sql.functions import when, col

# Add runway_category column
df_combined = (
    df_combined
    .withColumn(
        "runway_category",
        when(col("length_ft") >= 10000, "Long")
        .when((col("length_ft") >= 5000) & (col("length_ft") < 10000), "Medium")
        .otherwise("Short")
    )
)

# Preview result
df_combined.select("ident", "airport_ident", "length_ft", "runway_category").show(10)

+-----+-------------+---------+---------------+
|ident|airport_ident|length_ft|runway_category|
+-----+-------------+---------+---------------+
|  00A|          00A|       80|          Short|
| 00AK|         00AK|     2500|          Short|
| 00AL|         00AL|     2100|          Short|
| 00AN|         00AN|     4517|          Short|
| 00AS|         00AS|     1450|          Short|
| 00AZ|         00AZ|     1700|          Short|
| 00CA|         00CA|     6000|         Medium|
| 00FA|         00FA|     3200|          Short|
| 00FL|         00FL|     4700|          Short|
| 00GA|         00GA|     2600|          Short|
+-----+-------------+---------+---------------+
only showing top 10 rows


In [0]:
# Create temporary view
df_combined.createOrReplaceTempView("airport_view")

spark.sql("""
    SELECT
        iso_country,
        COUNT(*) AS runway_count
    FROM airport_view
    GROUP BY iso_country
    ORDER BY runway_count DESC
    LIMIT 10
""").show()

+-----------+------------+
|iso_country|runway_count|
+-----------+------------+
|         US|       26450|
|         BR|        5825|
|         AU|        2103|
|         CA|        1578|
|         AR|         761|
|         FR|         659|
|         GB|         600|
|         DE|         592|
|         RU|         489|
|         ID|         414|
+-----------+------------+



In [0]:
spark.sql("""
    SELECT
        continent,
        AVG(length_ft) AS avg_runway_length
    FROM airport_view
    WHERE length_ft IS NOT NULL
    GROUP BY continent
    ORDER BY avg_runway_length DESC
""").show()

+---------+------------------+
|continent| avg_runway_length|
+---------+------------------+
|       AN| 7768.857142857143|
|       AS|6925.6469996647675|
|       AF| 6787.526448362721|
|       EU| 4103.151335311572|
|       OC|3558.0401785714284|
|       SA|2857.0891141239963|
|       NA| 2599.724218941758|
+---------+------------------+



In [0]:
spark.sql("""
    SELECT
        name,
        municipality,
        COUNT(*) AS lighted_runway_count
    FROM airport_view
    WHERE iso_country = 'IN'
      AND lighted = '1'
    GROUP BY name, municipality
    HAVING COUNT(*) >= 1
    ORDER BY lighted_runway_count DESC
""").show()

+--------------------+------------+--------------------+
|                name|municipality|lighted_runway_count|
+--------------------+------------+--------------------+
|Indira Gandhi Int...|   New Delhi|                   3|
|Navi Mumbai Inter...| Navi Mumbai|                   2|
|Raja Bhoj Interna...|      Bhopal|                   2|
|Rajiv Gandhi Inte...|   Hyderabad|                   2|
|Netaji Subhash Ch...|     Kolkata|                   2|
|Agra Airport / Ag...|        Agra|                   2|
|   Prayagraj Airport|   Allahabad|                   2|
|Ambala Air Force ...|      Ambala|                   2|
|Chennai Internati...|     Chennai|                   2|
|Kempegowda Intern...|   Bengaluru|                   2|
|Mangaluru Interna...|   Mangaluru|                   2|
|Bidar Airport / B...|       Bidar|                   2|
|Chhatrapati Shiva...|      Mumbai|                   2|
|Tambaram Air Forc...|     Chennai|                   2|
|       Jammu Airport|       Ja

In [0]:
spark.sql("""
    SELECT
        name,
        iso_country,
        MAX(length_ft) AS max_runway_length
    FROM airport_view
    GROUP BY name, iso_country
    ORDER BY max_runway_length DESC
    LIMIT 5
""").show()

+--------------------+-----------+-----------------+
|                name|iso_country|max_runway_length|
+--------------------+-----------+-----------------+
|Gunflint Seaplane...|         US|            30000|
|Libby Camps Seapl...|         US|            26000|
|Brookville Reserv...|         US|            25000|
|Long Lake Seaplan...|         US|            25000|
|Conchas Lake Seap...|         US|            21120|
+--------------------+-----------+-----------------+



In [0]:
# Path to save Parquet file
parquet_path = "/Volumes/vita_prac/vita_sche/vita_vol/df_combined_parquet"

# Remove duplicate column names (keeps all airport columns + selected runway columns)
df_combined_clean = df_combined.select(
    "ident",
    "type",
    "name",
    "latitude_deg",
    "longitude_deg",
    "elevation_ft",
    "continent",
    "iso_country",
    "iso_region",
    "municipality",
    "scheduled_service",
    "gps_code",
    "iata_code",
    "local_code",
    "home_link",
    "wikipedia_link",
    "keywords",
    "airport_ident",
    "length_ft",
    "width_ft",
    "surface",
    "lighted",
    "closed",
    "le_ident",
    "le_latitude_deg",
    "le_longitude_deg",
    "le_elevation_ft",
    "le_heading_degT",
    "le_displaced_threshold_ft",
    "he_ident",
    "he_latitude_deg",
    "he_longitude_deg",
    "he_elevation_ft",
    "he_heading_degT",
    "he_displaced_threshold_ft",
    "runway_category"
)

# Save cleaned DataFrame as Parquet
df_combined_clean.write.mode("overwrite").parquet(parquet_path)


In [0]:
# Read Parquet file back
df_parquet = spark.read.parquet(parquet_path)

# Verify schema and first 10 rows
df_parquet.printSchema()
df_parquet.show(10)

root
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = true)
 |-- surface: string (nullable = true)
 |-- lighted: string (nullable = true)
 |-- closed: string (nullable = true)
 |-- le_ident: string (nullable = true)